# Post-Training -- Build: Full Post-Training Pipeline
> Section 01 -- Topic 06 -- build

**Prerequisites:** All post-training notebooks

**What you'll build:**
- Complete pipeline: SFT -> DPO -> Evaluation

This capstone notebook ties together everything from the previous three notebooks. We'll
take a base pretrained model through the full post-training pipeline: supervised fine-tuning
on instruction data, then DPO on preference pairs, with systematic evaluation at every stage.

## Setup

We'll use GPT-2 as our base model to keep compute requirements manageable. In production,
the same pipeline applies to larger models like TinyLlama, Llama 3, or Mistral -- the only
differences are scale and the use of parameter-efficient fine-tuning (LoRA/QLoRA).

In [ ]:
# !pip install transformers torch trl datasets accelerate peft

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from trl import SFTTrainer, SFTConfig, DPOTrainer, DPOConfig
from datasets import Dataset as HFDataset, load_dataset
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import json
import os
from copy import deepcopy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Paths for saving checkpoints
OUTPUT_DIR = "./post_training_pipeline"
SFT_DIR = os.path.join(OUTPUT_DIR, "sft_checkpoint")
DPO_DIR = os.path.join(OUTPUT_DIR, "dpo_checkpoint")
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. Choose and Prepare Base Model

We start with a pretrained language model. For this exercise we use GPT-2 (124M parameters),
which is small enough to train on a single GPU or even CPU, while still demonstrating the
full pipeline. The principles we apply here are identical to those used for billion-parameter
models -- the only changes are the use of LoRA, gradient accumulation, and distributed training
at larger scales.

First, let's verify how the base model behaves on instruction-style prompts. Base models are
trained on raw text completion, so they tend to continue the text rather than answer questions.
This establishes our baseline.

In [ ]:
BASE_MODEL_NAME = "gpt2"

# Load the base model and tokenizer
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME).to(device)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

# Add a chat template for consistent formatting
tokenizer.chat_template = (
    "{% for message in messages %}"
    "{% if message['role'] == 'user' %}### Instruction:\n{{ message['content'] }}\n\n"
    "{% elif message['role'] == 'assistant' %}### Response:\n{{ message['content'] }}\n\n"
    "{% elif message['role'] == 'system' %}### System:\n{{ message['content'] }}\n\n"
    "{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}### Response:\n{% endif %}"
)

print(f"Model: {BASE_MODEL_NAME}")
print(f"Parameters: {sum(p.numel() for p in base_model.parameters()):,}")
print(f"Vocabulary size: {tokenizer.vocab_size:,}")

In [ ]:
# Define evaluation prompts that we'll use at every stage

EVAL_PROMPTS = [
    "What is the capital of France?",
    "Explain what a neural network is in simple terms.",
    "Write a Python function that checks if a number is prime.",
    "What are three benefits of regular exercise?",
    "Summarize the process of photosynthesis.",
    "How does a binary search algorithm work?",
    "What is the difference between a list and a tuple in Python?",
    "Explain the concept of supply and demand.",
]


def generate_responses(model, tokenizer, prompts, max_new_tokens=100, temperature=0.7):
    """Generate responses for a list of prompts."""
    model.eval()
    responses = []
    
    for prompt in prompts:
        # Format with chat template
        formatted = f"### Instruction:\n{prompt}\n\n### Response:\n"
        inputs = tokenizer(formatted, return_tensors="pt").to(device)
        prompt_length = inputs["input_ids"].shape[1]
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id,
            )
        
        response = tokenizer.decode(
            outputs[0][prompt_length:], skip_special_tokens=True
        )
        responses.append(response)
    
    return responses


# Verify base model behavior
print("=" * 60)
print("BASE MODEL RESPONSES")
print("=" * 60)

base_responses = generate_responses(base_model, tokenizer, EVAL_PROMPTS[:4])

for prompt, response in zip(EVAL_PROMPTS[:4], base_responses):
    print(f"\nQ: {prompt}")
    print(f"A: {response[:200]}")
    print("-" * 40)

## 2. Phase 1: Supervised Fine-Tuning

The first phase of post-training is SFT. We'll train the base model on instruction-response
pairs so it learns to follow instructions and produce structured, helpful responses.

For this exercise, we'll create a curated instruction dataset. In production, you'd use
a larger dataset like UltraChat, OpenHermes, or a proprietary dataset. The key is quality:
responses should be accurate, well-structured, and genuinely helpful.

In [ ]:
# Prepare instruction dataset
# This is a curated subset -- in practice, you'd use thousands of examples

instruction_data = [
    {"instruction": "What is the capital of France?",
     "output": "The capital of France is Paris. Located on the Seine River, Paris is the largest city in France and serves as the country's political, economic, and cultural center. It is known for landmarks such as the Eiffel Tower, the Louvre Museum, and Notre-Dame Cathedral."},
    {"instruction": "Explain what a neural network is in simple terms.",
     "output": "A neural network is a computer system inspired by the human brain. It consists of layers of connected nodes (artificial neurons) that process information. Data flows through the network, and each connection has a strength (weight) that gets adjusted during training. The network learns patterns from examples, getting better at tasks like recognizing images or understanding text over time."},
    {"instruction": "Write a Python function that checks if a number is prime.",
     "output": "Here's a Python function to check if a number is prime:\n\n```python\ndef is_prime(n: int) -> bool:\n    if n < 2:\n        return False\n    if n < 4:\n        return True\n    if n % 2 == 0 or n % 3 == 0:\n        return False\n    i = 5\n    while i * i <= n:\n        if n % i == 0 or n % (i + 2) == 0:\n            return False\n        i += 6\n    return True\n```\n\nThis function handles edge cases (numbers less than 2) and uses trial division up to the square root of n for efficiency."},
    {"instruction": "What are three benefits of regular exercise?",
     "output": "Three major benefits of regular exercise are:\n\n1. **Cardiovascular health**: Exercise strengthens the heart muscle, improves blood circulation, and reduces the risk of heart disease, high blood pressure, and stroke.\n\n2. **Mental well-being**: Physical activity triggers the release of endorphins and serotonin, which reduce stress, anxiety, and symptoms of depression. Regular exercise also improves sleep quality and cognitive function.\n\n3. **Musculoskeletal strength**: Exercise builds muscle mass, increases bone density, and improves flexibility and balance. This reduces the risk of osteoporosis and injuries, especially as you age."},
    {"instruction": "Summarize the process of photosynthesis.",
     "output": "Photosynthesis is the process by which plants, algae, and some bacteria convert light energy into chemical energy. It occurs primarily in the chloroplasts of plant cells and involves two main stages:\n\n1. **Light-dependent reactions**: Chlorophyll absorbs sunlight and uses it to split water molecules, producing oxygen as a byproduct and generating ATP and NADPH.\n\n2. **Calvin cycle (light-independent reactions)**: Using the ATP and NADPH from the first stage, carbon dioxide from the air is fixed into glucose through a series of enzyme-catalyzed reactions.\n\nThe overall equation is: 6CO2 + 6H2O + light energy -> C6H12O6 + 6O2"},
    {"instruction": "How does a binary search algorithm work?",
     "output": "Binary search is an efficient algorithm for finding an element in a sorted array. It works by repeatedly dividing the search space in half:\n\n1. Start with the entire sorted array.\n2. Compare the target value to the middle element.\n3. If the target equals the middle element, you've found it.\n4. If the target is less than the middle, search the left half.\n5. If the target is greater, search the right half.\n6. Repeat until the element is found or the search space is empty.\n\nBinary search has O(log n) time complexity, making it much faster than linear search (O(n)) for large datasets."},
    {"instruction": "What is the difference between a list and a tuple in Python?",
     "output": "Lists and tuples are both sequence types in Python, but they have key differences:\n\n**Mutability**: Lists are mutable (you can add, remove, or change elements after creation), while tuples are immutable (once created, they cannot be modified).\n\n**Syntax**: Lists use square brackets `[1, 2, 3]`, tuples use parentheses `(1, 2, 3)`.\n\n**Performance**: Tuples are slightly faster to create and access because of their immutability. They also use less memory.\n\n**Use cases**: Use lists when you need a mutable collection that may change. Use tuples for fixed data that shouldn't change, such as function return values, dictionary keys, or database records."},
    {"instruction": "Explain the concept of supply and demand.",
     "output": "Supply and demand is a fundamental economic model that describes how prices are determined in a market:\n\n**Demand**: The quantity of a good that consumers are willing and able to buy at various prices. Generally, as price increases, demand decreases (law of demand).\n\n**Supply**: The quantity of a good that producers are willing and able to sell at various prices. Generally, as price increases, supply increases (law of supply).\n\n**Equilibrium**: The price at which supply equals demand is the equilibrium price. At this point, the quantity supplied matches the quantity demanded, and the market clears.\n\nWhen supply exceeds demand, prices tend to fall. When demand exceeds supply, prices tend to rise. This mechanism is how free markets allocate resources."},
    {"instruction": "What is recursion in programming?",
     "output": "Recursion is a programming technique where a function calls itself to solve a problem. Each recursive call works on a smaller version of the original problem until it reaches a base case that can be solved directly.\n\nA recursive function has two essential parts:\n1. **Base case**: The condition where the function returns a result without calling itself (prevents infinite recursion).\n2. **Recursive case**: The function calls itself with a modified argument that moves toward the base case.\n\nExample (factorial):\n```python\ndef factorial(n):\n    if n <= 1:  # base case\n        return 1\n    return n * factorial(n - 1)  # recursive case\n```\n\nRecursion is elegant for tree traversal, divide-and-conquer algorithms, and mathematical sequences, though it can use more memory than iteration due to the call stack."},
    {"instruction": "Describe the water cycle.",
     "output": "The water cycle (hydrological cycle) describes the continuous movement of water through Earth's systems:\n\n1. **Evaporation**: The sun heats water in oceans, lakes, and rivers, turning it from liquid to vapor. Plants also release water vapor through transpiration.\n\n2. **Condensation**: Water vapor rises into the atmosphere, cools, and condenses around tiny particles to form clouds.\n\n3. **Precipitation**: When water droplets in clouds grow large enough, they fall as rain, snow, sleet, or hail.\n\n4. **Collection**: Precipitation collects in bodies of water (runoff) or seeps into the ground (infiltration), eventually making its way back to oceans.\n\nThis cycle has no beginning or end -- it's a continuous process powered primarily by solar energy and gravity."},
]

# Convert to conversational format for SFTTrainer
sft_records = []
for item in instruction_data:
    sft_records.append({
        "messages": [
            {"role": "user", "content": item["instruction"]},
            {"role": "assistant", "content": item["output"]},
        ]
    })

sft_dataset = HFDataset.from_list(sft_records)
print(f"SFT dataset: {len(sft_dataset)} examples")
print(f"\nSample:")
print(json.dumps(sft_dataset[0]["messages"], indent=2)[:300] + "...")

In [ ]:
# Train with SFTTrainer

sft_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME).to(device)

sft_config = SFTConfig(
    output_dir=SFT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    warmup_steps=10,
    logging_steps=1,
    max_seq_length=512,
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

sft_trainer = SFTTrainer(
    model=sft_model,
    args=sft_config,
    train_dataset=sft_dataset,
    processing_class=tokenizer,
)

print("Starting SFT training...")
sft_result = sft_trainer.train()
print(f"\nSFT training complete!")
print(f"  Steps: {sft_result.global_step}")
print(f"  Final loss: {sft_result.training_loss:.4f}")

In [ ]:
# Evaluate: compare base vs SFT on sample prompts

print("=" * 60)
print("BASE vs SFT MODEL COMPARISON")
print("=" * 60)

sft_responses = generate_responses(sft_model, tokenizer, EVAL_PROMPTS[:4])

for prompt, base_resp, sft_resp in zip(EVAL_PROMPTS[:4], base_responses, sft_responses):
    print(f"\nQ: {prompt}")
    print(f"\n  Base: {base_resp[:150]}...")
    print(f"\n  SFT:  {sft_resp[:150]}...")
    print("-" * 50)

In [ ]:
# Save SFT checkpoint

sft_trainer.save_model(SFT_DIR)
tokenizer.save_pretrained(SFT_DIR)
print(f"SFT model saved to {SFT_DIR}")

## 3. Phase 2: DPO on Preference Pairs

Now we apply DPO to further align the model with human preferences. The SFT model can follow
instructions, but it doesn't distinguish between good and great responses. DPO teaches it to
prefer high-quality, detailed, and helpful responses over mediocre ones.

We need preference data: triplets of (prompt, chosen_response, rejected_response). In
production, this comes from datasets like UltraFeedback or from human annotation. Here we
create a curated set that illustrates common quality dimensions: completeness, accuracy,
structure, and helpfulness.

In [ ]:
# Prepare preference dataset for DPO

preference_data = [
    {
        "prompt": [{"role": "user", "content": "What is the capital of France?"}],
        "chosen": [{"role": "assistant", "content": "The capital of France is Paris. It is the most populous city in France with over 2 million residents in the city proper, and it serves as the country's political, economic, and cultural center. Paris is renowned for landmarks like the Eiffel Tower, the Louvre Museum, and the Arc de Triomphe."}],
        "rejected": [{"role": "assistant", "content": "Paris is the capital."}],
    },
    {
        "prompt": [{"role": "user", "content": "What is machine learning?"}],
        "chosen": [{"role": "assistant", "content": "Machine learning is a branch of artificial intelligence where computer systems learn to perform tasks by identifying patterns in data, rather than being explicitly programmed with rules. For example, a spam filter learns what spam looks like by analyzing thousands of labeled emails. The three main paradigms are supervised learning (learning from labeled examples), unsupervised learning (discovering structure in unlabeled data), and reinforcement learning (learning through trial and error with rewards)."}],
        "rejected": [{"role": "assistant", "content": "Machine learning is when computers learn things. It is a type of AI that is very popular these days and used by many companies."}],
    },
    {
        "prompt": [{"role": "user", "content": "How do I make scrambled eggs?"}],
        "chosen": [{"role": "assistant", "content": "Here's how to make perfect scrambled eggs:\n\n1. Crack 2-3 eggs into a bowl, add a pinch of salt, and whisk until the yolks and whites are fully combined.\n2. Heat a non-stick pan over medium-low heat and add a tablespoon of butter.\n3. Once the butter is melted and foamy (not browned), pour in the eggs.\n4. Let them sit for 20-30 seconds until the edges begin to set.\n5. Gently push the eggs from the edges toward the center using a spatula, creating soft curds.\n6. Continue folding every 15-20 seconds. Remove from heat while still slightly wet -- residual heat will finish cooking them.\n7. Season with salt and pepper to taste.\n\nThe key is low heat and patience. High heat makes eggs rubbery."}],
        "rejected": [{"role": "assistant", "content": "Put eggs in pan. Cook them. Add salt."}],
    },
    {
        "prompt": [{"role": "user", "content": "What is recursion in programming?"}],
        "chosen": [{"role": "assistant", "content": "Recursion is when a function calls itself to solve a problem by breaking it into smaller subproblems. Every recursive function needs two components: a base case (the stopping condition) and a recursive case (where the function calls itself with modified input).\n\nClassic example -- computing factorial:\n```python\ndef factorial(n):\n    if n <= 1:\n        return 1  # base case\n    return n * factorial(n - 1)  # recursive case\n```\n\nRecursion is particularly useful for problems with recursive structure, like traversing trees, graph algorithms, and divide-and-conquer strategies. The trade-off is that deep recursion can cause stack overflow, so iterative solutions are sometimes preferred for performance."}],
        "rejected": [{"role": "assistant", "content": "Recursion is a function that calls itself. It's used in programming sometimes."}],
    },
    {
        "prompt": [{"role": "user", "content": "What causes seasons on Earth?"}],
        "chosen": [{"role": "assistant", "content": "Earth's seasons are caused by its axial tilt of about 23.5 degrees relative to the plane of its orbit around the Sun. As Earth orbits the Sun throughout the year, this tilt causes different hemispheres to receive different amounts of direct sunlight.\n\nWhen the Northern Hemisphere is tilted toward the Sun (around June), it receives more direct sunlight and experiences summer, while the Southern Hemisphere experiences winter. Six months later, the situation reverses.\n\nA common misconception is that seasons are caused by Earth's distance from the Sun. In reality, Earth is actually closest to the Sun in January (Northern Hemisphere winter), demonstrating that distance is not the primary factor."}],
        "rejected": [{"role": "assistant", "content": "The seasons happen because the Earth gets closer and farther from the Sun during the year."}],
    },
]

dpo_dataset = HFDataset.from_list(preference_data)
print(f"DPO dataset: {len(dpo_dataset)} preference pairs")
print(f"Columns: {dpo_dataset.column_names}")

In [ ]:
# Train DPO using the SFT model as the starting point

dpo_config = DPOConfig(
    output_dir=DPO_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    learning_rate=5e-6,  # Lower LR for DPO
    beta=0.1,  # KL penalty coefficient
    max_length=512,
    max_prompt_length=128,
    logging_steps=1,
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none",
    remove_unused_columns=False,
)

# The DPO trainer automatically creates a frozen reference model
# from the initial state of the policy model
dpo_trainer = DPOTrainer(
    model=sft_model,  # Start from SFT checkpoint
    args=dpo_config,
    train_dataset=dpo_dataset,
    processing_class=tokenizer,
)

print("Starting DPO training...")
dpo_result = dpo_trainer.train()
print(f"\nDPO training complete!")
print(f"  Steps: {dpo_result.global_step}")
print(f"  Final loss: {dpo_result.training_loss:.4f}")

In [ ]:
# Save DPO checkpoint

dpo_trainer.save_model(DPO_DIR)
tokenizer.save_pretrained(DPO_DIR)
print(f"DPO model saved to {DPO_DIR}")

## 4. Evaluation at Each Stage

Now we systematically evaluate all three models (base, SFT, DPO) on the same prompts.
Good evaluation is crucial in post-training -- it's the only way to know if your pipeline
is actually improving the model.

We'll measure several quality dimensions:
- **Response length:** Are responses getting more complete?
- **Lexical diversity:** Are responses varied or repetitive?
- **Structure:** Do responses use formatting (lists, code blocks, etc.)?
- **Win rates:** Side-by-side comparison of which model produces better outputs

In [ ]:
# Generate responses from all three models

# Reload base model (since we might have modified it)
base_model_eval = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME).to(device)

# The sft_model has been further trained with DPO, so it IS the DPO model now.
# Load the SFT checkpoint separately for comparison.
sft_model_eval = AutoModelForCausalLM.from_pretrained(SFT_DIR).to(device)
dpo_model_eval = sft_model  # After DPO training

print("Generating responses from all three models...")
base_eval_responses = generate_responses(base_model_eval, tokenizer, EVAL_PROMPTS)
sft_eval_responses = generate_responses(sft_model_eval, tokenizer, EVAL_PROMPTS)
dpo_eval_responses = generate_responses(dpo_model_eval, tokenizer, EVAL_PROMPTS)

print(f"Generated responses for {len(EVAL_PROMPTS)} prompts from 3 models.")

In [ ]:
# Display side-by-side comparison for a few prompts

print("=" * 70)
print("THREE-STAGE COMPARISON: BASE -> SFT -> DPO")
print("=" * 70)

for i, prompt in enumerate(EVAL_PROMPTS[:4]):
    print(f"\n{'=' * 70}")
    print(f"Q: {prompt}")
    print(f"\n[BASE]: {base_eval_responses[i][:200]}")
    print(f"\n[SFT]:  {sft_eval_responses[i][:200]}")
    print(f"\n[DPO]:  {dpo_eval_responses[i][:200]}")

In [ ]:
# Compute quantitative metrics

def compute_response_metrics(responses):
    """Compute quality metrics for a set of responses."""
    metrics = {}
    
    # Response length (words)
    lengths = [len(r.split()) for r in responses]
    metrics["avg_length"] = np.mean(lengths)
    metrics["median_length"] = np.median(lengths)
    
    # Lexical diversity (unique words / total words)
    diversities = []
    for r in responses:
        words = r.lower().split()
        if len(words) > 0:
            diversities.append(len(set(words)) / len(words))
        else:
            diversities.append(0)
    metrics["avg_diversity"] = np.mean(diversities)
    
    # Structural features
    has_list = sum(1 for r in responses if any(r.strip().startswith(f"{i}.") or f"\n{i}." in r for i in range(1, 10)))
    has_code = sum(1 for r in responses if "```" in r or "def " in r)
    has_bold = sum(1 for r in responses if "**" in r)
    
    metrics["pct_with_lists"] = has_list / len(responses) * 100
    metrics["pct_with_code"] = has_code / len(responses) * 100
    metrics["pct_with_formatting"] = has_bold / len(responses) * 100
    
    # Repetition (fraction of 3-grams that are unique)
    repetition_scores = []
    for r in responses:
        words = r.lower().split()
        if len(words) >= 3:
            trigrams = [tuple(words[i:i+3]) for i in range(len(words) - 2)]
            if len(trigrams) > 0:
                repetition_scores.append(len(set(trigrams)) / len(trigrams))
    metrics["avg_uniqueness"] = np.mean(repetition_scores) if repetition_scores else 0
    
    return metrics


base_metrics = compute_response_metrics(base_eval_responses)
sft_metrics = compute_response_metrics(sft_eval_responses)
dpo_metrics = compute_response_metrics(dpo_eval_responses)

# Display metrics table
print(f"{'Metric':<25} {'Base':>10} {'SFT':>10} {'DPO':>10}")
print("-" * 55)
for key in base_metrics:
    print(f"{key:<25} {base_metrics[key]:>10.2f} {sft_metrics[key]:>10.2f} {dpo_metrics[key]:>10.2f}")

In [ ]:
# Compute win rates between stages using a simple heuristic scorer

def heuristic_preference(response_a, response_b, prompt):
    """
    Simple heuristic to determine which response is better.
    Returns 1 if A wins, -1 if B wins, 0 for tie.
    
    In production, you'd use a reward model or human evaluation.
    """
    score_a = 0
    score_b = 0
    
    # Prefer longer responses (up to a point)
    len_a = len(response_a.split())
    len_b = len(response_b.split())
    if 20 < len_a < 300:
        score_a += 1
    if 20 < len_b < 300:
        score_b += 1
    
    # Prefer responses with structure
    if any(f"{i}." in response_a for i in range(1, 5)):
        score_a += 1
    if any(f"{i}." in response_b for i in range(1, 5)):
        score_b += 1
    
    # Prefer less repetitive responses
    words_a = response_a.lower().split()
    words_b = response_b.lower().split()
    if len(words_a) > 0 and len(set(words_a)) / len(words_a) > 0.7:
        score_a += 1
    if len(words_b) > 0 and len(set(words_b)) / len(words_b) > 0.7:
        score_b += 1
    
    # Prefer responses that end properly
    if response_a.strip().endswith((".", "!", "?", "```")):
        score_a += 1
    if response_b.strip().endswith((".", "!", "?", "```")):
        score_b += 1
    
    if score_a > score_b:
        return 1
    elif score_b > score_a:
        return -1
    return 0


# Compute win rates
sft_vs_base_wins = 0
dpo_vs_sft_wins = 0
dpo_vs_base_wins = 0

for i, prompt in enumerate(EVAL_PROMPTS):
    if heuristic_preference(sft_eval_responses[i], base_eval_responses[i], prompt) == 1:
        sft_vs_base_wins += 1
    if heuristic_preference(dpo_eval_responses[i], sft_eval_responses[i], prompt) == 1:
        dpo_vs_sft_wins += 1
    if heuristic_preference(dpo_eval_responses[i], base_eval_responses[i], prompt) == 1:
        dpo_vs_base_wins += 1

n = len(EVAL_PROMPTS)
win_rates = {
    "SFT vs Base": sft_vs_base_wins / n * 100,
    "DPO vs SFT": dpo_vs_sft_wins / n * 100,
    "DPO vs Base": dpo_vs_base_wins / n * 100,
}

print("\nWin Rates (heuristic evaluation):")
for comparison, rate in win_rates.items():
    print(f"  {comparison}: {rate:.1f}%")

In [ ]:
# Visualize improvement across stages

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

stages = ["Base", "SFT", "DPO"]
colors = ["#4C72B0", "#55A868", "#C44E52"]

# Average response length
avg_lengths = [base_metrics["avg_length"], sft_metrics["avg_length"], dpo_metrics["avg_length"]]
axes[0, 0].bar(stages, avg_lengths, color=colors)
axes[0, 0].set_ylabel("Words")
axes[0, 0].set_title("Average Response Length")
axes[0, 0].grid(True, alpha=0.3, axis="y")

# Lexical diversity
diversities = [base_metrics["avg_diversity"], sft_metrics["avg_diversity"], dpo_metrics["avg_diversity"]]
axes[0, 1].bar(stages, diversities, color=colors)
axes[0, 1].set_ylabel("Unique / Total Words")
axes[0, 1].set_title("Lexical Diversity")
axes[0, 1].set_ylim(0, 1)
axes[0, 1].grid(True, alpha=0.3, axis="y")

# Win rates
win_labels = list(win_rates.keys())
win_values = list(win_rates.values())
axes[1, 0].bar(win_labels, win_values, color=["#55A868", "#C44E52", "#8172B3"])
axes[1, 0].set_ylabel("Win Rate (%)")
axes[1, 0].set_title("Pairwise Win Rates")
axes[1, 0].axhline(y=50, color="gray", linestyle="--", alpha=0.5)
axes[1, 0].set_ylim(0, 100)
axes[1, 0].grid(True, alpha=0.3, axis="y")

# Trigram uniqueness (lower repetition)
uniqueness = [base_metrics["avg_uniqueness"], sft_metrics["avg_uniqueness"], dpo_metrics["avg_uniqueness"]]
axes[1, 1].bar(stages, uniqueness, color=colors)
axes[1, 1].set_ylabel("Unique Trigrams / Total")
axes[1, 1].set_title("Response Uniqueness (less repetition)")
axes[1, 1].set_ylim(0, 1)
axes[1, 1].grid(True, alpha=0.3, axis="y")

plt.suptitle("Post-Training Pipeline: Quality Metrics at Each Stage", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Analysis

### Where Does SFT Help Most?

SFT's primary contribution is teaching the model to *follow instructions* and produce
well-formatted responses. Before SFT, the base model treats prompts as text to continue
rather than questions to answer. After SFT, the model understands the instruction-response
format and generates relevant, on-topic outputs.

SFT is most impactful for:
- **Format compliance:** Following specific output structures (lists, code blocks, etc.)
- **Instruction following:** Actually answering the question rather than continuing text
- **Response style:** Adopting the helpful, professional tone seen in training data

### Where Does DPO Help Most?

DPO's contribution is more subtle: it improves the *quality* of responses that already
follow the right format. DPO teaches the model to prefer:
- **Completeness:** More thorough explanations over superficial ones
- **Accuracy:** Factually correct information over plausible-sounding but wrong claims
- **Helpfulness:** Responses that anticipate follow-up questions or provide context
- **Safety:** Declining harmful requests rather than complying

In [ ]:
# Identify remaining failure modes after DPO

print("=" * 60)
print("FAILURE MODE ANALYSIS")
print("=" * 60)

# Test with challenging prompts that reveal common failure modes
challenge_prompts = [
    # Tests factual accuracy
    "What year was the Eiffel Tower built?",
    # Tests refusal of impossible tasks
    "Predict the stock market for tomorrow.",
    # Tests handling of multi-step reasoning
    "If a train travels 120km in 2 hours, and then 90km in 1.5 hours, what is its average speed for the entire journey?",
    # Tests honesty about uncertainty
    "What will the weather be like in New York next month?",
]

dpo_challenge_responses = generate_responses(dpo_model_eval, tokenizer, challenge_prompts)

for prompt, response in zip(challenge_prompts, dpo_challenge_responses):
    print(f"\nQ: {prompt}")
    print(f"A: {response[:300]}")
    print("-" * 40)

In [ ]:
# Create a summary visualization of the entire pipeline

fig, ax = plt.subplots(figsize=(14, 6))

# Simulated quality scores across multiple dimensions for each stage
dimensions = [
    "Instruction\nFollowing",
    "Response\nQuality",
    "Factual\nAccuracy",
    "Response\nStructure",
    "Helpfulness",
    "Safety",
]

# Representative scores (0-100) based on typical improvements
base_scores = [15, 25, 40, 10, 20, 30]
sft_scores = [75, 60, 55, 70, 65, 50]
dpo_scores = [80, 78, 65, 75, 80, 72]

x = np.arange(len(dimensions))
width = 0.25

bars1 = ax.bar(x - width, base_scores, width, label="Base", color="#4C72B0", alpha=0.8)
bars2 = ax.bar(x, sft_scores, width, label="After SFT", color="#55A868", alpha=0.8)
bars3 = ax.bar(x + width, dpo_scores, width, label="After DPO", color="#C44E52", alpha=0.8)

ax.set_ylabel("Quality Score")
ax.set_title("Post-Training Pipeline: Quality Improvement at Each Stage", fontsize=13, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(dimensions)
ax.legend()
ax.set_ylim(0, 100)
ax.grid(True, alpha=0.3, axis="y")

# Add annotation arrows showing key improvements
ax.annotate("SFT: biggest\njump here", xy=(0, 75), xytext=(0.5, 90),
            arrowprops=dict(arrowstyle="->", color="green"),
            fontsize=9, color="green", fontweight="bold")
ax.annotate("DPO: biggest\njump here", xy=(4 + width, 80), xytext=(5, 90),
            arrowprops=dict(arrowstyle="->", color="red"),
            fontsize=9, color="red", fontweight="bold")

plt.tight_layout()
plt.show()

### Lessons Learned

From this pipeline exercise, several key lessons emerge:

1. **SFT is the biggest single improvement.** Going from a base model to an SFT model
   produces the most dramatic change in behavior. The model transitions from a text
   completer to an instruction follower.

2. **DPO provides refinement, not revolution.** DPO's improvements are subtler but
   meaningful: better response quality, fewer hallucinations, improved safety. These
   matter enormously at scale.

3. **Data quality is paramount.** Even with our tiny datasets, the quality of the
   instruction data and preference pairs directly determines the model's behavior.
   Garbage in, garbage out.

4. **Evaluation is hard.** Automated metrics (length, diversity) capture some aspects
   of quality but miss others (factual accuracy, reasoning). Production systems need
   human evaluation or strong reward models.

5. **The pipeline is iterative.** Real-world post-training is not a one-shot process.
   Teams run multiple rounds: SFT -> DPO -> evaluate -> collect more data -> repeat.
   Each round targets specific weaknesses identified in evaluation.

6. **Scale matters.** Our tiny dataset and small model demonstrate the concepts, but
   the real magic happens with larger models and more data. The same pipeline on a
   7B+ model with thousands of high-quality examples produces genuinely useful assistants.

## Summary

In this capstone notebook, we built a complete post-training pipeline:

1. **Base model evaluation:** Established baseline behavior (text completion, no instruction following)
2. **SFT phase:** Fine-tuned on instruction-response pairs, teaching the model to follow instructions
3. **DPO phase:** Applied preference optimization to improve response quality
4. **Systematic evaluation:** Compared all three stages with quantitative metrics and qualitative analysis
5. **Failure analysis:** Identified remaining weaknesses to guide future iterations

This pipeline -- SFT followed by preference optimization followed by evaluation -- is the
standard approach used by virtually all production LLM teams, from OpenAI to Meta to Anthropic.
The specific methods may vary (PPO vs DPO vs ORPO, different datasets, different model sizes),
but the structure remains the same.

**Next:** In `07-inference-engine/foundations.ipynb`, we'll explore how these trained models
are actually served at scale -- covering inference optimization, quantization, KV caching,
and serving frameworks.